In [1]:
# Importing all necessary libraries

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Read the Excel file

In [2]:
data = pd.read_excel('course_recommendation.xlsx')

In [3]:
data.head()

,courseid,title,technology,topic,abstract,url,image_url
0,1,Categorical Data in the Tidyverse,R,Data Manipulation,NaN,https://www.datacamp.com/courses/categorical-d...,NaN
1,2,GARCH Models in R,R,Applied Finance,NaN,https://www.datacamp.com/courses/garch-models-...,NaN
2,3,Structural Equation Modeling with lavaan in R,R,Probability & Statistics,NaN,https://www.datacamp.com/courses/structural-eq...,NaN
3,4,Analyzing IoT Data in Python,Python,Data Manipulation,NaN,https://www.datacamp.com/courses/analyzing-iot...,NaN
4,5,Introduction to Spark SQL in Python,Python,Data Manipulation,NaN,https://www.datacamp.com/courses/introduction-...,NaN


#### Description of the data
## The file contains 4 features about the course which have been uploaded into our database
* title - Title of the Course
* technology - Technology of the course 
* topic - course topic
* abstract - Short summary of the course
* url - Course url - if applicable

In [4]:
# Copying the excel file for preprocessing

In [5]:
df = data.copy()

In [6]:
df.fillna(" ",inplace=True)

In [7]:
df.head()

,courseid,title,technology,topic,abstract,url,image_url
0,1,Categorical Data in the Tidyverse,R,Data Manipulation,,https://www.datacamp.com/courses/categorical-d...,
1,2,GARCH Models in R,R,Applied Finance,,https://www.datacamp.com/courses/garch-models-...,
2,3,Structural Equation Modeling with lavaan in R,R,Probability & Statistics,,https://www.datacamp.com/courses/structural-eq...,
3,4,Analyzing IoT Data in Python,Python,Data Manipulation,,https://www.datacamp.com/courses/analyzing-iot...,
4,5,Introduction to Spark SQL in Python,Python,Data Manipulation,,https://www.datacamp.com/courses/introduction-...,


In [8]:
# List of course wrt topic  ( Unique value count of courses based on topic)
print(f"There are {len(df['topic'].value_counts())} types of courses based on topics\n")
print("*"*55)
print('\033[1m' + 'List of courses wrt topic:' + '\033[0m')
print(df['topic'].value_counts())


There are 13 types of courses based on topics

*******************************************************
List of courses wrt topic:
                             154
Probability & Statistics      57
Machine Learning              57
Programming                   50
Data Manipulation             33
Data Visualization            22
Applied Finance               18
Case Studies                  12
Importing & Cleaning Data      8
Other                          7
Reporting                      5
Data Engineering               2
Management                     1
Name: topic, dtype: int64


# Objective of the Project
* Recommend top 10 courses (course name and course link ) based on the user's browse history / search history 
* The 'tags' field is used to fetch the similar courses 

In [9]:
# The 4  columns 'technology' , topic' and 'abstract'  contain important keywords that the user might be using to search the course. Hence three fields are combined together to create a new field 'tags' to help us to recommend the appropriate courses 
df['tag'] = df['title'] + " " + df['technology'] +" " + df['topic'] + " "+df['abstract']

In [10]:
df.head()

,courseid,title,technology,topic,abstract,url,image_url,tag
0,1,Categorical Data in the Tidyverse,R,Data Manipulation,,https://www.datacamp.com/courses/categorical-d...,,Categorical Data in the Tidyverse R Data Manip...
1,2,GARCH Models in R,R,Applied Finance,,https://www.datacamp.com/courses/garch-models-...,,GARCH Models in R R Applied Finance
2,3,Structural Equation Modeling with lavaan in R,R,Probability & Statistics,,https://www.datacamp.com/courses/structural-eq...,,Structural Equation Modeling with lavaan in R ...
3,4,Analyzing IoT Data in Python,Python,Data Manipulation,,https://www.datacamp.com/courses/analyzing-iot...,,Analyzing IoT Data in Python Python Data Manip...
4,5,Introduction to Spark SQL in Python,Python,Data Manipulation,,https://www.datacamp.com/courses/introduction-...,,Introduction to Spark SQL in Python Python Dat...


In [11]:
df.shape

(426, 8)

In [12]:
# Required fields - title, tag and url
# All other fields can be dropped

df1 = df[['title','tag','url']]

In [13]:
df1.head()

,title,tag,url
0,Categorical Data in the Tidyverse,Categorical Data in the Tidyverse R Data Manip...,https://www.datacamp.com/courses/categorical-d...
1,GARCH Models in R,GARCH Models in R R Applied Finance,https://www.datacamp.com/courses/garch-models-...
2,Structural Equation Modeling with lavaan in R,Structural Equation Modeling with lavaan in R ...,https://www.datacamp.com/courses/structural-eq...
3,Analyzing IoT Data in Python,Analyzing IoT Data in Python Python Data Manip...,https://www.datacamp.com/courses/analyzing-iot...
4,Introduction to Spark SQL in Python,Introduction to Spark SQL in Python Python Dat...,https://www.datacamp.com/courses/introduction-...


# Text Preprocessing

In [14]:
# Convert all tags into lowercase

df1['tag'] = df1['tag'].apply(lambda x:x.lower())

C:\Users\Suhas\AppData\Local\Temp\ipykernel_19448\3301979309.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['tag'] = df1['tag'].apply(lambda x:x.lower())


In [15]:
# Data cleaning - Removing all punctuation marks and stop words from tags 

import nltk
import string

from nltk.corpus import stopwords

def data_clean(text):
    result = []
    words = nltk.word_tokenize(text)
    for word in words:
        if (word not in string.punctuation) & (word not in stopwords.words('english')):
            result.append(word)
    return " ".join(result)

In [16]:
df1['tag'] = df1['tag'].apply(data_clean)

C:\Users\Suhas\AppData\Local\Temp\ipykernel_19448\1487538441.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['tag'] = df1['tag'].apply(data_clean)


In [17]:
df1.head()

,title,tag,url
0,Categorical Data in the Tidyverse,categorical data tidyverse r data manipulation,https://www.datacamp.com/courses/categorical-d...
1,GARCH Models in R,garch models r r applied finance,https://www.datacamp.com/courses/garch-models-...
2,Structural Equation Modeling with lavaan in R,structural equation modeling lavaan r r probab...,https://www.datacamp.com/courses/structural-eq...
3,Analyzing IoT Data in Python,analyzing iot data python python data manipula...,https://www.datacamp.com/courses/analyzing-iot...
4,Introduction to Spark SQL in Python,introduction spark sql python python data mani...,https://www.datacamp.com/courses/introduction-...


In [21]:
# Lemmatization - Converting text into standard basic word

from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemma = WordNetLemmatizer()

def lemmatizing (text):
    lemma_data =[]
    for word in text.split():
        lemma_data.append(lemma.lemmatize(word))
    return " ".join(lemma_data)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Suhas\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [22]:
df1['tag'] = df1['tag'].apply(lemmatizing)

LookupError: 
**********************************************************************
  Resource [93momw-1.4[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('omw-1.4')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/omw-1.4[0m

  Searched in:
    - 'C:\\Users\\Suhas/nltk_data'
    - 'C:\\Users\\Suhas\\anaconda3\\nltk_data'
    - 'C:\\Users\\Suhas\\anaconda3\\share\\nltk_data'
    - 'C:\\Users\\Suhas\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\Suhas\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [149]:
df1.head()

,title,tag,url
0,Categorical Data in the Tidyverse,rdata manipulation,https://www.datacamp.com/courses/categorical-d...
1,GARCH Models in R,rapplied finance,https://www.datacamp.com/courses/garch-models-...
2,Structural Equation Modeling with lavaan in R,rprobability statistic,https://www.datacamp.com/courses/structural-eq...
3,Analyzing IoT Data in Python,pythondata manipulation,https://www.datacamp.com/courses/analyzing-iot...
4,Introduction to Spark SQL in Python,pythondata manipulation,https://www.datacamp.com/courses/introduction-...


In [150]:
df1['index'] = df1.index

C:\Users\Suhas\AppData\Local\Temp/ipykernel_7984/712662445.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['index'] = df1.index


In [151]:
df1 = df1[['index','title','tag','url']]

In [152]:
df1.head()

,index,title,tag,url
0,0,Categorical Data in the Tidyverse,rdata manipulation,https://www.datacamp.com/courses/categorical-d...
1,1,GARCH Models in R,rapplied finance,https://www.datacamp.com/courses/garch-models-...
2,2,Structural Equation Modeling with lavaan in R,rprobability statistic,https://www.datacamp.com/courses/structural-eq...
3,3,Analyzing IoT Data in Python,pythondata manipulation,https://www.datacamp.com/courses/analyzing-iot...
4,4,Introduction to Spark SQL in Python,pythondata manipulation,https://www.datacamp.com/courses/introduction-...


In [153]:
# Vectorization - converting 'tags' columns into vectors
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

tags_vectors = cv.fit_transform(df1['tag'])

In [154]:
# Similarity measure - Calculating the similarity of tags of each course with itself and all other courses 
# This is used to find how similar the courses are with each other

from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tags_vectors)

In [165]:
# Input the keyword for course recommendationcharacter animation
keyword = input("Enter the keyword: ").lower()

Enter the keyword: DDOS


In [166]:
# Finding the course index of the course based on keyword search

def get_index_from_tag(keyword):
    if len(df1[df1['tag'].str.contains(keyword)].index )!=0:
        return df1[df1['tag'].str.contains(keyword)].index[0]
    else:
        return 'Null'


In [167]:
course_index = get_index_from_tag(keyword)
course_index

303

In [168]:
# Sorting all courses with the measure of similarity (Descending order)
if course_index != 'Null':
    similar_courses  = list(enumerate(similarity[course_index]))
    similar_courses = sorted(similar_courses,reverse=True,key=lambda x:x[1])

In [169]:
# Recommending top 10 courses with highest similarity score
if course_index != 'Null':
    for i in range(0, len(similar_courses[:10])):
        print(f"{i+1}. {df1.iloc[similar_courses[i][0]].title}\t {df1.iloc[similar_courses[i][0]].url}")
else:
    print('Sorry, No match found. Please enter another keyword')

1. Security and Privacy in TOR Network	 https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=467035.13810980272&type=2&murl=https%3A%2F%2Fwww.coursera.org%2Flearn%2Fsecurity-and-privacy-in-tor-network
2. Advanced System Security Design	 https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=467035.14041349568&type=2&murl=https%3A%2F%2Fwww.coursera.org%2Fspecializations%2Fadvanced-system-security-design
3. IT Fundamentals for Cybersecurity	 https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=467035.14475568802&type=2&murl=https%3A%2F%2Fwww.coursera.org%2Fspecializations%2Fit-fundamentals-cybersecurity
4. how to kick-start cds preparation	 https://upscfever.com/upsc-fever/en/how-to-study/en-how2-cds.html
5. Site Reliability Engineering: Measuring and Managing Reliability	 https://click.linksynergy.com/link?id=16zYSsTLZqk&offerid=467035.13798362328&type=2&murl=https%3A%2F%2Fwww.coursera.org%2Flearn%2Fsite-reliability-engineering-slos
6. IT Security: Defense against the digit